# tensor unbind — procedural drill

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `tensor-unbind`. When a test cell passes, your progress is reported back to your account.

**What you'll practice.** Five unbind patterns that ramp from default-dim → explicit-dim → equivalence-with-`select` → tuple-destructure → ray-equation evaluation. Read the docstring, fill the function body, run the test cell. The solution sits in the collapsed `<details>` block below each exercise.

**Per-exercise structure** (Doughty et al. ACE 2024 — `[Bloom level] + [LO] + [Keywords] + [KCs]`):
Each exercise begins with a yaml block stating its Bloom cognitive level, learning objective, keywords, and the knowledge components (KCs) it targets. This makes the cognitive demand explicit instead of buried.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Indexing and selection` subtopic.
You can copy the token from your Delta Drills account page.

This drill exercises the **atom `tensor-unbind`**, which bridges to the bank subtopic `Numpy: Indexing and selection` for EWMA state. Completing all 5 exercises triggers a single `arena-rating` beacon at the end of the notebook.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "tensor-unbind"
DD_SUBTOPIC = "Numpy: Indexing and selection"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

# Track which exercises passed in this session.
_dd_passed = set()

## Unbind — quick refresher

**What it does.** `torch.unbind(x, dim=k)` returns a tuple of `x.shape[k]` tensors, each with axis `k` removed. The result is a *Python tuple* (not a tensor) of *views* (no copy).

**Default dim is 0.** `x.unbind()` peels along axis 0; `x.unbind(dim=1)` peels along axis 1.

**Idiomatic destructure.** `origin, direction = rays.unbind(dim=1)` is the canonical way to split a `(N, 2, 3)` rays tensor into two `(N, 3)` named components.

**Equivalence.** `x.unbind(dim=k)[i]` == `x.select(k, i)`. Prefer `select` for picking ONE slice; prefer `unbind` when you want ALL slices.

### Exercise 1 — unbind along default dim 0

> ```yaml
> Difficulty: ⚪⚪⚪⚪⚪
> Bloom level: Remember
> LO: Recall that `torch.unbind(x)` peels along `dim=0` into a tuple of `x.shape[0]` slices.
> Keywords: torch-unbind, default-dim, tuple-of-slices
> ```

**KCs targeted:** `unbind-default-dim`

Implement `ex1_unbind_rows(x)` to return a tuple of 1-D tensors — one for each row of the 2-D input `x`. Use `torch.unbind` with the default `dim=0`.

Input shape: `(R, C)`. Output: a tuple of `R` tensors, each of shape `(C,)`.

In [ ]:
def ex1_unbind_rows(x: Tensor) -> tuple:
    """Unbind a 2-D tensor along dim 0 → tuple of rows."""
    raise NotImplementedError()


def _test_ex1():
    x = t.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0],
                  [7.0, 8.0, 9.0]])
    out = ex1_unbind_rows(x)
    assert isinstance(out, tuple), f'expected tuple, got {type(out).__name__}'
    assert len(out) == 3, f'expected 3 rows, got {len(out)}'
    for i, row in enumerate(out):
        assert row.shape == (3,), f'row {i} shape {row.shape}, expected (3,)'
        assert t.allclose(row, x[i]), f'row {i} mismatch'
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_unbind_rows(x: Tensor) -> tuple:
    return t.unbind(x)
```

**Default `dim=0`.** `t.unbind(x)` is shorthand for `t.unbind(x, dim=0)`. The result is a tuple — NOT a list, NOT a tensor — because unbind always returns a Python sequence of views over storage, never a single tensor.
</details>

### Exercise 2 — unbind along an explicit axis

> ```yaml
> Difficulty: 🔴⚪⚪⚪⚪
> Bloom level: Apply
> LO: Apply the `dim=` argument to unbind along a non-default axis.
> Keywords: dim-arg, column-unbind, axis-aware
> ```

**KCs targeted:** `unbind-explicit-dim`

Implement `ex2_unbind_columns(x)` to return a tuple of column tensors from a 2-D input.

Input shape: `(R, C)`. Output: a tuple of `C` tensors, each of shape `(R,)`. Use `dim=1`.

In [ ]:
def ex2_unbind_columns(x: Tensor) -> tuple:
    """Unbind a 2-D tensor along dim 1 → tuple of columns."""
    raise NotImplementedError()


def _test_ex2():
    x = t.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])
    out = ex2_unbind_columns(x)
    assert isinstance(out, tuple), f'expected tuple, got {type(out).__name__}'
    assert len(out) == 3, f'expected 3 columns, got {len(out)}'
    for j, col in enumerate(out):
        assert col.shape == (2,), f'col {j} shape {col.shape}, expected (2,)'
        assert t.allclose(col, x[:, j]), f'col {j} mismatch'
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_unbind_columns(x: Tensor) -> tuple:
    return t.unbind(x, dim=1)
```

**`dim=1` on a `(R, C)` tensor** peels along the column axis, yielding `C` tensors each of shape `(R,)`. The shape drops the unbound axis — `(R, C)` → `C` × `(R,)`.
</details>

### Exercise 3 — unbind equivalence with `.select`

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply the equivalence `unbind(x, dim=k)[i] == x.select(k, i)` to pick a single slice via `select`.
> Keywords: select, equivalence, view-semantics
> ```

**KCs targeted:** `unbind-equiv-to-slice`

Implement `ex3_select_via_unbind(x, dim, i)`: return the `i`-th slice along `dim`, using only `torch.unbind` and Python tuple indexing.

Functionally equivalent to `x.select(dim, i)` — the point is to make the equivalence concrete.

Inputs: any-shape tensor `x`, integer `dim`, integer `i` in `[0, x.shape[dim])`.

In [ ]:
def ex3_select_via_unbind(x: Tensor, dim: int, i: int) -> Tensor:
    """Return the i-th slice along `dim`, implemented via unbind."""
    raise NotImplementedError()


def _test_ex3():
    x = t.randn(4, 5, 6)
    assert t.allclose(ex3_select_via_unbind(x, 0, 2), x.select(0, 2))
    assert t.allclose(ex3_select_via_unbind(x, 1, 0), x.select(1, 0))
    assert t.allclose(ex3_select_via_unbind(x, 2, 5), x.select(2, 5))
    # Output shape must drop the selected axis.
    out = ex3_select_via_unbind(x, 1, 3)
    assert out.shape == (4, 6), f'expected (4,6), got {tuple(out.shape)}'
    _dd_passed.add('ex3')
    print("ex3 ✓")

_test_ex3()

<details><summary>Solution</summary>

```python
def ex3_select_via_unbind(x: Tensor, dim: int, i: int) -> Tensor:
    return t.unbind(x, dim=dim)[i]
```

**Why this matters.** `unbind` is a *view* op — each output tensor shares storage with `x`. It's not slower than `select` for picking one slice; the cost is the Python tuple construction. Prefer `select` when you want ONE slice; prefer `unbind` when you want ALL slices (so you don't loop with `select` N times).
</details>

### Exercise 4 — destructure a fixed number of slices

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply tuple unpacking to give names to the components of a small-axis unbind.
> Keywords: destructure, xyz, fixed-arity
> ```

**KCs targeted:** `unbind-tuple-destructure`

Implement `ex4_xyz_components(v)`: given a 1-D 3-vector `v`, return `x + y + z` (the sum of its three components), but extract them via tuple unpacking from `v.unbind()`.

Example: `v = tensor([1.0, 2.0, 3.0])` → `x, y, z = v.unbind()` → return `x + y + z` = 6.0 (as a 0-D tensor).

The point is the idiom `x, y, z = v.unbind()` — replaces the clunky `v[0], v[1], v[2]` form when you want named scalar slices.

In [ ]:
def ex4_xyz_components(v: Tensor) -> Tensor:
    """Destructure a 3-vector via unbind, return x + y + z."""
    raise NotImplementedError()


def _test_ex4():
    v = t.tensor([1.0, 2.0, 3.0])
    out = ex4_xyz_components(v)
    assert out.dim() == 0, f'expected scalar (0-D), got shape {tuple(out.shape)}'
    assert t.allclose(out, t.tensor(6.0)), f'expected 6.0, got {out.item()}'

    v2 = t.tensor([-1.0, 0.5, 2.5])
    assert t.allclose(ex4_xyz_components(v2), t.tensor(2.0))
    _dd_passed.add('ex4')
    print("ex4 ✓")

_test_ex4()

<details><summary>Solution</summary>

```python
def ex4_xyz_components(v: Tensor) -> Tensor:
    x, y, z = v.unbind()
    return x + y + z
```

**Why named destructure helps.** Adding three components is trivial either way, but consider Phong shading: `ka * Ia + kd * Id * dot(N, L) + ks * Is * dot(R, V)^n`. Having `N, L, R, V = vectors.unbind()` early makes the formula readable; threading `vectors[0], vectors[1], ...` through obscures the math.
</details>

### Exercise 5 — decompose rays, evaluate at parameter t

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Create
> LO: Synthesize axis-explicit unbind + tuple destructure to split a rays tensor and evaluate the parametric ray equation.
> Keywords: ray-tracing, origin-direction, param-evaluate, multi-kc
> ```

**KCs targeted:** `unbind-explicit-dim`, `unbind-tuple-destructure`, `unbind-ray-decomposition`

Implement `ex5_evaluate_rays(rays, t_param)`. The canonical Ray Tracing use of `unbind`:

Input `rays`: shape `(N, 2, 3)`. Each ray is a `(2, 3)` block where row 0 is the origin and row 1 is the direction (in 3-D).

Output: `(N, 3)` — for each ray, return `origin + t_param * direction`.

Implementation: use `rays.unbind(dim=1)` to peel out origin and direction as `(N, 3)` tensors, then return the parametric evaluation.

> ⚠️ **Integrative exercise.** Combines 3 KCs (explicit dim, tuple destructure, ray decomposition). Empirical work (Lohr et al. ITiCSE 2025) shows 3-concept exercises drop to ~40% solvability — expect a step up vs Exercises 1-4.

In [ ]:
def ex5_evaluate_rays(rays: Tensor, t_param: float) -> Tensor:
    """Given rays of shape (N, 2, 3) where rays[i] = [origin, direction],
    return origin + t_param * direction for each ray. Output: (N, 3).
    """
    raise NotImplementedError()


def _test_ex5():
    rays = t.tensor([
        # ray 0 — at origin, direction +x
        [[0.0, 0.0, 0.0], [1.0, 0.0, 0.0]],
        # ray 1 — at (1, 2, 3), direction +y
        [[1.0, 2.0, 3.0], [0.0, 1.0, 0.0]],
        # ray 2 — at (0, 0, 1), direction (1, 1, 0)
        [[0.0, 0.0, 1.0], [1.0, 1.0, 0.0]],
    ])
    out = ex5_evaluate_rays(rays, t_param=2.0)
    assert out.shape == (3, 3), f'expected (3,3), got {tuple(out.shape)}'
    expected = t.tensor([
        [2.0, 0.0, 0.0],   # 0 + 2*x = (2, 0, 0)
        [1.0, 4.0, 3.0],   # (1,2,3) + 2*(0,1,0) = (1, 4, 3)
        [2.0, 2.0, 1.0],   # (0,0,1) + 2*(1,1,0) = (2, 2, 1)
    ])
    assert t.allclose(out, expected), f'value mismatch:\n{out}\nvs\n{expected}'
    # t=0 should return origins unchanged.
    out0 = ex5_evaluate_rays(rays, t_param=0.0)
    assert t.allclose(out0, rays[:, 0, :]), 't=0 must return origins'
    _dd_passed.add('ex5')
    print("ex5 ✓")

_test_ex5()

<details><summary>Solution</summary>

```python
def ex5_evaluate_rays(rays: Tensor, t_param: float) -> Tensor:
    origin, direction = rays.unbind(dim=1)
    return origin + t_param * direction
```

**The parametric ray equation: `r(t) = o + t·d`.** This is the most common Ray Tracing computation — given an array of rays and a hit parameter, find the world-space hit point. `unbind(dim=1)` is the cleanest split: `origin` and `direction` come out as named `(N, 3)` tensors with no shape arithmetic.

**Compare:** `rays[:, 0, :] + t * rays[:, 1, :]` works and is equivalent, but the unbind version reads like math. Prefer it when your variables have physical meaning.
</details>

## Done

Run the cell below to report your progress to Delta Drills. The beacon fires only if all 5 exercises passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1', 'ex2', 'ex3', 'ex4', 'ex5'}

def _dd_feedback_level(num_passed: int) -> str:
    """Map exercise-pass count → arena-rating feedback enum."""
    if num_passed == 5: return 'not_much'
    if num_passed >= 3: return 'somewhat'
    return 'a_lot'

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {len(missing)} exercises still failing: {sorted(missing)}.")
        print("[Delta Drills] not reporting until all 5 pass.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}',
        'subtopics': [DD_SUBTOPIC],
        'feedback': _dd_feedback_level(len(_dd_passed)),
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()